# 📦 Exercício: o robô no armazém

O **RL-1** (o robô do exemplo 1) foi transferido para um **armazém**: precisa levar uma caixa da garagem `S` até a **doca de carga** `D`. Agora é **você** quem programa o ambiente, o ciclo agente–ambiente e a política.

| Conceito | No armazém |
|---|---|
| **Estado** | a posição do robô `(linha, coluna)` |
| **Ação** | `cima`, `baixo`, `esquerda`, `direita` |
| **Recompensa** | **−1 por passo** (energia gasta) |
| **Episódio** | começa em `S` e termina na doca `D` |

### Como funciona
- ⏱️ **~20 minutos.** Os exercícios 1 a 4 são o núcleo; o 5 é bônus.
- Em cada exercício você completa as linhas marcadas com ✏️ e roda a célula **🧪 Teste**, que diz ✅ ou ❌.
- Travou? Abra a **Dica 1** (a ideia). Ainda travado? A **Dica 2** tem o código.
- Os desenhos já estão prontos (célula recolhida): o foco é RL, não gráfico.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrow
from collections import deque

np.random.seed(0)
ACOES = ['cima', 'baixo', 'esquerda', 'direita']
MOV = {'cima': (-1, 0), 'baixo': (1, 0), 'esquerda': (0, -1), 'direita': (0, 1)}

# S = garagem (início) · . = corredor · # = prateleira · D = doca de carga (fim)
MAPA = [
    "S..#....",
    ".#.#.##.",
    ".#...#..",
    ".####.#.",
    "......#D",
]

In [ ]:
# @title Desenho e funções de apoio (rode e siga em frente)
def desenhar(amb, robo=None, rastro=None, politica=None, titulo=None, ax=None, marcar=None):
    """Desenha o armazém. politica: função estado -> ação (desenha uma seta por casa)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    nl = amb.n_linhas
    xy = lambda l, c: (c + .5, nl - 1 - l + .5)
    for l in range(nl):
        for c in range(amb.n_colunas):
            ch = amb.mapa[l][c]
            cor = {'#': '#8D6E63', 'D': '#C8E6C9', 'S': '#FFF8E1'}.get(ch, '#FAFAFA')
            ax.add_patch(Rectangle((c, nl - 1 - l), 1, 1, fc=cor, ec='#D7CCC8', lw=1))
            x, y = xy(l, c)
            if ch == 'D':
                ax.add_patch(Rectangle((x - .22, y - .2), .44, .36, fc='#A1887F', ec='#4E342E', zorder=3))
                ax.text(x, y + .3, 'doca', ha='center', fontsize=7, color='#2E7D32')
            if ch == 'S':
                ax.text(c + .08, nl - 1 - l + .82, 'garagem', fontsize=6, color='#8D6E63')
            if marcar is not None and (l, c) in marcar:
                ax.add_patch(Rectangle((c, nl - 1 - l), 1, 1, fc='#66BB6A' if marcar[(l, c)] else '#EF5350',
                                       alpha=.45, zorder=2))
            if politica is not None and ch not in '#D':
                try:
                    a = politica((l, c))
                except Exception:
                    a = None
                if a in MOV:
                    dl, dc = MOV[a]
                    ax.add_patch(FancyArrow(x - dc * .22, y + dl * .22, dc * .38, -dl * .38, width=.05,
                                            head_width=.2, head_length=.14, color='#E65100', zorder=7))
    if rastro:
        pts = [xy(*s) for s in rastro]
        ax.plot([p[0] for p in pts], [p[1] for p in pts], '-', color='#1565C0', alpha=.35, lw=5, zorder=8)
    if robo is not None:
        x, y = xy(*robo)
        ax.add_patch(FancyBboxPatch((x - .27, y - .25), .54, .46, boxstyle='round,pad=0.02,rounding_size=0.12',
                                    fc='#1565C0', ec='#0D2A4A', lw=1.5, zorder=10))
        for dx in (-.11, .11):
            ax.add_patch(Circle((x + dx, y + .02), .075, fc='white', zorder=11))
            ax.add_patch(Circle((x + dx, y + .02), .035, fc='#111', zorder=12))
    ax.set_xlim(0, amb.n_colunas); ax.set_ylim(0, nl); ax.set_aspect('equal'); ax.axis('off')
    if titulo:
        ax.set_title(titulo, fontsize=11, fontweight='bold')
    plt.show()


def politica_otima(amb):
    """Referência: anda sempre para a casa vizinha mais perto da doca (busca em largura no mapa)."""
    dist = {amb.doca: 0}
    fila = deque([amb.doca])
    while fila:
        l, c = fila.popleft()
        for dl, dc in MOV.values():
            v = (l + dl, c + dc)
            if amb.livre(*v) and v not in dist:
                dist[v] = dist[(l, c)] + 1
                fila.append(v)
    pol = lambda s: min(ACOES, key=lambda a: dist.get((s[0] + MOV[a][0], s[1] + MOV[a][1]), 999))
    return pol, dist


def politica_aleatoria(estado):
    return np.random.choice(ACOES)


def ok(cond, msg_ok, msg_erro):
    print(('✅ ' + msg_ok) if cond else ('❌ ' + msg_erro))
    return cond

---
## Exercício 1 — O ambiente: complete o `step()`

Todo ambiente de RL "conversa" com o agente por dois comandos (slide 39): `reset()` começa o episódio e `step(acao)` devolve `(novo_estado, recompensa, fim)`. O `reset()` já está pronto — complete o `step()`:

1. calcule a posição para onde o robô **tentaria** ir;
2. se a casa estiver livre, ele anda; se for prateleira ou borda, **fica onde está**;
3. a recompensa é **−1** em todo passo, e o episódio termina quando o robô chega à doca.

In [ ]:
class Armazem:
    def __init__(self, mapa):
        self.mapa = [list(linha) for linha in mapa]
        self.n_linhas, self.n_colunas = len(mapa), len(mapa[0])
        self.inicio, self.doca = self.achar('S'), self.achar('D')
        self.reset()

    def achar(self, ch):
        return next((l, c) for l in range(self.n_linhas) for c in range(self.n_colunas) if self.mapa[l][c] == ch)

    def livre(self, l, c):
        return 0 <= l < self.n_linhas and 0 <= c < self.n_colunas and self.mapa[l][c] != '#'

    def reset(self, inicio=None):
        self.estado = inicio or self.inicio
        return self.estado

    def step(self, acao):
        dl, dc = MOV[acao]

        # ✏️ 1) posição para onde o robô TENTARIA ir
        l, c = None, None

        # ✏️ 2) se (l, c) estiver livre, o robô anda: atualize self.estado
        #       (se não estiver livre, não faça nada: ele fica onde está)

        # ✏️ 3) recompensa e fim do episódio
        recompensa = None
        fim = None
        return self.estado, recompensa, fim


armazem = Armazem(MAPA)
desenhar(armazem, robo=armazem.inicio, titulo='O armazém')

In [ ]:
# 🧪 Teste do Exercício 1
try:
    a = Armazem(MAPA)
    s, r, fim = a.step('direita')
    ok(s == (0, 1) and r == -1 and fim is False, "andou para a direita: ((0, 1), -1, False)",
       f"de (0, 0) para a direita deveria dar ((0, 1), -1, False) e deu {(s, r, fim)}")
    a.reset(); s, r, _ = a.step('cima')
    ok(s == (0, 0) and r == -1, "na borda, ficou no lugar e ainda pagou −1",
       f"subindo de (0, 0) o robô deveria ficar em (0, 0) com recompensa −1; deu {(s, r)}")
    a.estado = (0, 2); s, r, _ = a.step('direita')
    ok(s == (0, 2) and r == -1, "bateu na prateleira, ficou no lugar e pagou −1",
       f"(0, 3) é prateleira: deveria ficar em (0, 2) com −1; deu {(s, r)}")
    a.estado = (3, 7); s, r, fim = a.step('baixo')
    ok(s == (4, 7) and fim is True, "chegou à doca e o episódio terminou",
       f"de (3, 7) para baixo chega à doca: esperado ((4, 7), -1, True), deu {(s, r, fim)}")
except Exception as e:
    print("❌ O step() ainda não funciona:", type(e).__name__, e)

<details><summary>💡 Dica 1 — a ideia</summary>

1) Some o deslocamento `(dl, dc)` à posição atual `self.estado`.
2) Use o método `self.livre(l, c)` num `if`.
3) A recompensa é sempre −1; o fim é a comparação do estado com `self.doca`.
</details>

<details><summary>🔑 Dica 2 — o código</summary>

```python
l, c = self.estado[0] + dl, self.estado[1] + dc
if self.livre(l, c):
    self.estado = (l, c)
recompensa = -1
fim = self.estado == self.doca
```
</details>

---
## Exercício 2 — O ciclo agente–ambiente e o retorno G

Agora escreva o **laço** que faz um episódio acontecer (slide 29): o agente observa o estado, a **política** escolhe a ação, o ambiente responde, e repete até o fim.

Depois, calcule o **retorno** $G$: a soma das recompensas do episódio. Como cada passo vale −1, $G = -(\text{número de passos})$.

In [ ]:
def rodar_episodio(amb, politica, max_passos=500, inicio=None):
    """Joga um episódio. Devolve a lista de estados visitados e a lista de recompensas."""
    estado = amb.reset(inicio)
    estados, recompensas = [estado], []
    for _ in range(max_passos):
        # ✏️ 1) a política escolhe a ação para o estado atual
        # ✏️ 2) o ambiente executa a ação (amb.step) e devolve estado, recompensa e fim
        # ✏️ 3) guarde o novo estado em `estados` e a recompensa em `recompensas`
        # ✏️ 4) se o episódio terminou, pare o laço
        pass
    return estados, recompensas


def retorno(recompensas):
    # ✏️ G = soma das recompensas
    return None


np.random.seed(1)
estados, recompensas = rodar_episodio(armazem, politica_aleatoria)
print(f"Episódio aleatório: {len(recompensas)} passos | G = {retorno(recompensas)}")
desenhar(armazem, robo=estados[-1], rastro=estados, titulo='Por onde o robô aleatório andou')

In [ ]:
# 🧪 Teste do Exercício 2
try:
    ok(retorno([-1, -1, -1]) == -3, "retorno([-1, -1, -1]) = -3", f"retorno([-1, -1, -1]) deveria ser -3 e deu {retorno([-1, -1, -1])}")
    est, rec = rodar_episodio(Armazem(MAPA), lambda s: 'cima', max_passos=50)
    ok(len(rec) == 50 and set(est) == {(0, 0)}, "uma política que só sobe fica presa por 50 passos",
       f"sempre 'cima' deveria dar 50 passos parados em (0, 0); deu {len(rec)} passos")
    otima, dist = politica_otima(armazem)
    est, rec = rodar_episodio(Armazem(MAPA), otima)
    ok(est[-1] == armazem.doca and retorno(rec) == -dist[armazem.inicio],
       f"com a política ótima: chegou em {len(rec)} passos, G = {retorno(rec)}",
       f"a política ótima deveria chegar com G = {-dist[armazem.inicio]}; deu {retorno(rec)} (você parou o laço no fim?)")
except Exception as e:
    print("❌ Ainda não funciona:", type(e).__name__, e)

<details><summary>💡 Dica 1 — a ideia</summary>

Dentro do `for`: chame `politica(estado)`, depois `amb.step(acao)` (ele devolve **três** coisas), use `.append` nas duas listas e `break` quando `fim` for verdadeiro.
O retorno é só a soma da lista: o Python tem `sum`.
</details>

<details><summary>🔑 Dica 2 — o código</summary>

```python
# dentro do for:
acao = politica(estado)
estado, recompensa, fim = amb.step(acao)
estados.append(estado)
recompensas.append(recompensa)
if fim:
    break

# na função retorno:
return sum(recompensas)
```
</details>

---
## Exercício 3 — A sua política: uma tabela estado → ação

Uma política determinística é uma regra $a = \pi(s)$ (slide 34). A forma mais direta de escrevê-la é uma **tabela**: para cada casa, a ação.

Complete `minha_tabela` com o caminho da garagem até a doca. Casas que **não** estão na tabela usam `'cima'` — provavelmente uma péssima ideia, mas é o que acontece quando a política não sabe o que fazer.

> Guarde essa ideia: a **Q-table** do Q-Learning é uma tabela parecida, mas preenchida sozinha, por tentativa e erro.

In [ ]:
minha_tabela = {
    (0, 0): 'direita',
    (0, 1): 'direita',
    # ✏️ continue o caminho, uma casa por linha, até chegar à doca (4, 7)
}


def minha_politica(estado):
    return minha_tabela.get(estado, 'cima')


est, rec = rodar_episodio(armazem, minha_politica, max_passos=100)
print(f"Sua política: {'chegou' if est[-1] == armazem.doca else 'NÃO chegou'} em {len(rec)} passos | G = {retorno(rec)}")
desenhar(armazem, robo=est[-1], rastro=est, politica=minha_politica, titulo='Sua política (setas) e o caminho do robô')

In [ ]:
# 🧪 Teste do Exercício 3
otima, dist = politica_otima(armazem)
est, rec = rodar_episodio(armazem, minha_politica, max_passos=100)
if ok(est[-1] == armazem.doca, f"chegou à doca em {len(rec)} passos", "o robô não chegou à doca: siga as setas no desenho e veja onde ele trava"):
    ok(len(rec) == dist[armazem.inicio], f"caminho mais curto possível ({dist[armazem.inicio]} passos)! 🏆",
       f"funciona, mas o mais curto tem {dist[armazem.inicio]} passos. Consegue achar?")

<details><summary>💡 Dica 1 — a ideia</summary>

Desenhe o caminho no mapa antes: da garagem, vá pela **linha de cima** até onde der, desça pelo meio e depois volte a subir para o lado direito.
Cuidado com o corredor de baixo: ele parece perto da doca, mas é um **beco sem saída**.
</details>

<details><summary>🔑 Dica 2 — o código</summary>

```python
minha_tabela = {
    (0, 0): 'direita', (0, 1): 'direita', (0, 2): 'baixo', (1, 2): 'baixo',
    (2, 2): 'direita', (2, 3): 'direita', (2, 4): 'cima', (1, 4): 'cima',
    (0, 4): 'direita', (0, 5): 'direita', (0, 6): 'direita', (0, 7): 'baixo',
    (1, 7): 'baixo', (2, 7): 'baixo', (3, 7): 'baixo',
}
```
</details>

---
## Exercício 4 — A competição (slide 131: um episódio não prova nada)

Complete `avaliar`: rode **n episódios** com a política e devolva a **taxa de chegada** e o **retorno médio**. Depois a célula põe as três políticas para competir.

> **Palpite antes de rodar:** quantos passos, em média, o robô aleatório gasta? Ele chega sempre (em até 200 passos)?

In [ ]:
def avaliar(amb, politica, n=200, max_passos=200):
    """Devolve (taxa de chegada à doca, retorno médio) em n episódios."""
    chegadas, retornos = [], []
    for _ in range(n):
        # ✏️ rode um episódio (rodar_episodio), anote se chegou à doca (True/False) e o retorno
        pass
    return np.mean(chegadas), np.mean(retornos)


np.random.seed(0)
otima, _ = politica_otima(armazem)
print(f"{'política':16s} {'chegou':>7s} {'retorno médio':>14s}")
for nome, pol in [('aleatória', politica_aleatoria), ('sua tabela', minha_politica), ('ótima', otima)]:
    taxa, G = avaliar(armazem, pol)
    print(f"{nome:16s} {taxa:7.0%} {G:14.1f}")

In [ ]:
# 🧪 Teste do Exercício 4
try:
    otima, dist = politica_otima(armazem)
    taxa, G = avaliar(armazem, otima, n=5)
    ok(taxa == 1 and G == -dist[armazem.inicio], f"política ótima: 100% de chegada e G = {G:.0f}",
       f"a política ótima deveria dar (1.0, {-dist[armazem.inicio]}); deu {(taxa, G)}")
    taxa, G = avaliar(armazem, lambda s: 'cima', n=5, max_passos=30)
    ok(taxa == 0 and G == -30, "política travada: 0% e G = −30 (cortada em 30 passos)",
       f"'sempre cima' com max_passos=30 deveria dar (0.0, -30); deu {(taxa, G)}")
except Exception as e:
    print("❌ Ainda não funciona:", type(e).__name__, e)

<details><summary>💡 Dica 1 — a ideia</summary>

É um laço de `n` episódios: em cada um, chame `rodar_episodio(amb, politica, max_passos)`. Chegou se o **último estado** da lista for `amb.doca`. O retorno vem da sua função `retorno`.
</details>

<details><summary>🔑 Dica 2 — o código</summary>

```python
for _ in range(n):
    est, rec = rodar_episodio(amb, politica, max_passos)
    chegadas.append(est[-1] == amb.doca)
    retornos.append(retorno(rec))
```
</details>

---
## ⭐ Exercício 5 (bônus) — O robô religou num lugar qualquer

Depois de uma queda de energia, o robô religa **numa casa qualquer** do armazém, não na garagem. A célula abaixo testa as duas políticas partindo de **cada** casa: verde = chegou à doca, vermelho = travou.

> **Palpite:** a partir de quantas casas a sua tabela consegue chegar?

In [ ]:
otima, _ = politica_otima(armazem)
casas = [(l, c) for l in range(armazem.n_linhas) for c in range(armazem.n_colunas)
         if armazem.livre(l, c) and (l, c) != armazem.doca]
for nome, pol in [('Sua tabela', minha_politica), ('Política ótima', otima)]:
    chegou = {s: rodar_episodio(armazem, pol, max_passos=100, inicio=s)[0][-1] == armazem.doca for s in casas}
    titulo = f"{nome}: chega à doca a partir de {sum(chegou.values())} de {len(casas)} casas"
    print(titulo)
    desenhar(armazem, politica=pol, marcar=chegou, titulo=titulo)

**O que aconteceu?** A sua tabela só sabe o que fazer **no caminho** que você escreveu. Fora dele, ela manda `'cima'` e o robô trava. A política ótima tem uma resposta para **todas** as casas.

Essa é a diferença entre **plano** e **política** dos slides 73–75: um roteiro decorado só funciona se tudo sair como previsto; uma política responde de **qualquer** lugar — como o GPS que recalcula a rota. **Desafio:** complete a sua tabela até todas as casas ficarem verdes.

---
## Reflexão final

1. No Exercício 1, bater na prateleira também custa −1. O que mudaria no comportamento de um agente que aprende se bater fosse **de graça**?
2. No Exercício 4, por que o robô aleatório demora tanto, se a doca está a 15 passos?
3. Você escreveu a tabela olhando o mapa. Um agente de RL **não vê o mapa**: como ele poderia preencher essa tabela sozinho, para **todas** as casas? (É o tema das próximas aulas: Q-Learning.)